In [2]:
pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 44.9 MB/s eta 0:00:00


In [9]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([('Burglary', 'Alarm'), ('Earthquake', 'Alarm'), ('Alarm', 'JohnCalls'), ('Alarm', 'MaryCalls')])

result1 = model.is_dconnected('Earthquake', 'Burglary', observed=['Alarm'])
print("Earthquake is independent of Burglary given Alarm:", not result1)

result2 = model.is_dconnected('Earthquake', 'Burglary', observed=[])
print("Earthquake is independent of Burglary Alarm:", not result2)

result3 = model.is_dconnected('JohnCalls', 'MaryCalls', observed=['Alarm'])
print("JohnCalls is independent of MarryCalls Given Alarm:", not result3)

result4 = model.is_dconnected('Earthquake', 'Burglary', observed=[])
print("JohnCalls is independent of MarryCalls given Alarm:", not result4)

Independencies = model.get_independencies()
print(Independencies)


Earthquake is independent of Burglary given Alarm: False
Earthquake is independent of Burglary Alarm: True
JohnCalls is independent of MarryCalls Given Alarm: True
JohnCalls is independent of MarryCalls given Alarm: True
(JohnCalls ⟂ Earthquake | Alarm)
(Earthquake ⟂ MaryCalls | Alarm)
(JohnCalls ⟂ MaryCalls | Alarm)
(MaryCalls ⟂ Burglary | Alarm)
(Earthquake ⟂ Burglary)
(JohnCalls ⟂ Burglary | Alarm)


In [10]:
# Define the CPDs

cpd_burglary = TabularCPD(variable='Burglary', variable_card=2, values=[[0.999], [0.001]])

cpd_earthquake = TabularCPD(variable='Earthquake', variable_card=2, values=[[0.998], [0.002]])

cpd_alarm = TabularCPD(variable='Alarm', variable_card=2,
                       values=[[0.999, 0.71, 0.06, 0.05],
                               [0.001, 0.29, 0.94, 0.95]],
                       evidence=['Burglary', 'Earthquake'],
                       evidence_card=[2, 2])

cpd_johncalls = TabularCPD(variable='JohnCalls', variable_card=2,
                            values=[[0.98, 0.1],
                                    [0.02, 0.9]],
                            evidence=['Alarm'],
                            evidence_card=[2])

cpd_marycalls = TabularCPD(variable='MaryCalls', variable_card=2,
                            values=[[0.99, 0.8],
                                    [0.01, 0.2]],
                            evidence=['Alarm'],
                            evidence_card=[2])

In [11]:
model.add_cpds(cpd_burglary, cpd_earthquake, cpd_alarm, cpd_johncalls, cpd_marycalls)

assert model.check_model()

print("Conditional Porbability Distributions (CPDs):")

for cpd in model.get_cpds():
    print(cpd)

Conditional Porbability Distributions (CPDs):
+-------------+-------+
| Burglary(0) | 0.999 |
+-------------+-------+
| Burglary(1) | 0.001 |
+-------------+-------+
+---------------+-------+
| Earthquake(0) | 0.998 |
+---------------+-------+
| Earthquake(1) | 0.002 |
+---------------+-------+
+------------+---------------+---------------+---------------+---------------+
| Burglary   | Burglary(0)   | Burglary(0)   | Burglary(1)   | Burglary(1)   |
+------------+---------------+---------------+---------------+---------------+
| Earthquake | Earthquake(0) | Earthquake(1) | Earthquake(0) | Earthquake(1) |
+------------+---------------+---------------+---------------+---------------+
| Alarm(0)   | 0.999         | 0.71          | 0.06          | 0.05          |
+------------+---------------+---------------+---------------+---------------+
| Alarm(1)   | 0.001         | 0.29          | 0.94          | 0.95          |
+------------+---------------+---------------+---------------+----------